#IPL Performance & Team Strategy Analysis (2008–2024)

## Project Overview
This notebook analyzes 16 years of IPL data (2008–2024) covering 1095 matches
and 260,000+ ball-by-ball deliveries. The goal is to think like a franchise consultant
— uncovering player value, toss impact, venue patterns, and team strategy using SQL.

## Tools Used
- **Python & Google Colab** — Environment
- **Pandas** — Loading and bridging CSV data
- **SQLite3** — In-memory SQL database for all analysis
- **Google Sheets** — Visualization (Phase 2)
- **Power BI** — Dashboard (Phase 3)

## Dataset
Source: Kaggle — IPL Complete Dataset 2008–2024
- `matches.csv` — 1095 rows, 20 columns (match-level data)
- `deliveries.csv` — 260,920 rows, 17 columns (ball-by-ball data)

In [ ]:
import pandas as pd
import sqlite3

## Step 1 — Loading the Data

We load both CSV files into pandas DataFrames, then push them into an
in-memory SQLite database. This lets us write real SQL queries without
needing any external database software.

- `matches` table → match results, venues, toss decisions, winners
- `deliveries` table → every ball bowled across all 1095 matches

In [ ]:
matches = pd.read_csv("/content/matches.csv")
deliveries = pd.read_csv("/content/deliveries.csv")

In [ ]:
conn = sqlite3.connect(":memory")
matches.to_sql("matches", conn, index=False, if_exists='replace')
deliveries.to_sql("deliveries", conn, index=False, if_exists='replace')

260920

In [ ]:
# Helper Function
def run_query(sql):
  return pd.read_sql_query(sql, conn)

In [ ]:
print("Done matches:", matches.shape, " | Deliviries:" , deliveries.shape)
print("matches column: ", matches.columns.tolist())
print("deliveries column: ", deliveries.columns.tolist())

Done matches: (1095, 20)  | Deliviries: (260920, 17)
matches column:  ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']
deliveries column:  ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


## Step 2 — SQL Analysis

All analysis is done using SQL via SQLite3. We explore 4 key business questions
that a franchise scout or team strategist would care about:

1. **Powerplay vs Death Overs** — Which batsmen perform best in high-pressure phases?
2. **Toss Impact by Venue** — Does winning the toss actually matter, and where?
3. **Bowler Economy: Finals vs League** — Who performs under pressure when it matters most?
4. **Home vs Away Performance** — Which teams rely on home advantage?

In [ ]:
# Query 1 — Top 10 Batsmen: Powerplay vs Death Overs
run_query("""
    SELECT
        batter,
        SUM(CASE WHEN over BETWEEN 1 AND 6 THEN batsman_runs ELSE 0 END) AS powerplay_runs,
        SUM(CASE WHEN over BETWEEN 16 AND 20 THEN batsman_runs ELSE 0 END) AS death_runs,
        ROUND(SUM(CASE WHEN over BETWEEN 1 AND 6 THEN batsman_runs ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN over BETWEEN 1 AND 6 THEN 1 ELSE 0 END), 0), 2) AS powerplay_sr,
        ROUND(SUM(CASE WHEN over BETWEEN 16 AND 20 THEN batsman_runs ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN over BETWEEN 16 AND 20 THEN 1 ELSE 0 END), 0), 2) AS death_sr
    FROM deliveries
    GROUP BY batter
    HAVING powerplay_runs > 300 AND death_runs > 200
    ORDER BY death_sr DESC
    LIMIT 10
""")

,batter,powerplay_runs,death_runs,powerplay_sr,death_sr
0,AB de Villiers,911,1421,114.59,223.78
1,RR Pant,526,626,121.48,196.86
2,CH Gayle,2314,404,138.81,193.30
3,V Kohli,2699,1099,115.15,192.47
4,JH Kallis,1010,303,102.33,190.57
5,F du Plessis,2215,416,141.71,189.09
6,RG Sharma,2082,1176,120.14,188.16
7,SA Yadav,980,516,134.80,186.96
8,KK Nair,570,211,121.54,185.09
9,JC Buttler,1559,412,137.36,183.93


**Key Insights**

AB de Villiers leads death overs with a strike rate of 223.78 across 1421 runs —
making him the most valuable finisher in IPL history by both volume and efficiency.
CH Gayle dominates the powerplay (2314 runs, 138.81 SR) but drops off in death overs,
confirming his role as a specialist opener rather than a finisher.

In [ ]:
# Query 2 — Win % by Toss Decision per Venue
run_query("""
    SELECT
         venue,
         toss_decision,
         count(*) AS total_matches,
         SUM(case when toss_winner = winner then 1 else 0 end) AS wins,
         ROUND(100.0 * SUM(case when toss_winner = winner then 1 else 0 end) / count(*), 2) AS wins_pct
    FROM matches
    GROUP BY venue, toss_decision
    HAVING total_matches >= 10
    ORDER BY wins_pct DESC
""")

,venue,toss_decision,total_matches,wins,wins_pct
0,Sawai Mansingh Stadium,field,28,19,67.86
1,Maharashtra Cricket Association Stadium,field,20,13,65.00
2,Sharjah Cricket Stadium,field,20,13,65.00
3,Eden Gardens,field,49,31,63.27
4,"MA Chidambaram Stadium, Chepauk",bat,34,21,61.76
5,Dr DY Patil Sports Academy,field,10,6,60.00
6,Kingsmead,bat,10,6,60.00
7,Subrata Roy Sahara Stadium,bat,15,9,60.00
8,"Wankhede Stadium, Mumbai",field,37,22,59.46
9,"Narendra Modi Stadium, Ahmedabad",field,19,11,57.89


**Key Insights**

Sawai Mansingh Stadium shows the strongest toss advantage — teams winning the toss
win 67.86% of matches there, suggesting strong pitch or dew factor influence.
Eden Gardens heavily favors toss winners choosing to field (63.27% win rate across 49 matches),
making it one of the most toss-sensitive venues in the IPL.

In [ ]:
# Query 3 — Bowler Economy: Finals vs League Stage
run_query("""
    SELECT
        d.bowler,
        ROUND(SUM(CASE when m.match_type = 'Final' then d.total_runs else 0 end) * 6.0 /
             NULLIF(SUM(CASE when m.match_type = 'Final' then 1 else 0 end), 0), 2) AS Finals_economy,
        ROUND(SUM(CASE when m.match_type != 'Final' then d.total_runs else 0 end) * 6.0 /
             NULLIF(SUM(case when m.match_type != 'Final' then 1 else 0 end), 0 ), 2) AS league_economy,
        COUNT(DISTINCT case when m.match_type = 'Final' then d.match_id end) AS finals_played
    FROM deliveries d
    JOIN matches m on d.match_id = m.id
    GROUP BY d.bowler
    HAVING finals_played >= 2
    ORDER BY finals_economy
    LIMIT 15
""")

,bowler,Finals_economy,league_economy,finals_played
0,AR Patel,4.53,7.32,2
1,TA Boult,5.40,8.18,2
2,KV Sharma,5.73,8.49,2
3,MJ McClenaghan,5.88,8.29,2
4,JD Unadkat,6.00,8.84,2
5,JJ Bumrah,6.25,7.26,3
6,JA Morkel,6.71,8.08,5
7,HH Pandya,6.86,8.96,4
8,B Kumar,6.87,7.47,3
9,Harbhajan Singh,6.91,7.04,4


**Key Insights**

AR Patel stands out as the best pressure bowler in IPL Finals — with an economy
of just 4.53 in finals compared to 7.32 in league stages, showing he significantly
elevates his performance when it matters most. JJ Bumrah and HH Pandya both maintain
sub-7 economy rates across 3-4 finals appearances, confirming their reputation as
match-winning bowlers in high-stakes games. Interestingly, SL Malinga's finals economy
(7.18) is actually higher than his league economy (7.03) — suggesting even legends
can struggle under final pressure.

In [ ]:
# Query 4 — Team Performance: Home vs Away
run_query("""
     SELECT
         team,
         SUM(home_match) AS Home_matches,
         SUM(home_win) AS Home_wins,
         ROUND(100.0 *SUM(home_win) / NULLIF(SUM(home_match), 0), 2) AS home_win_pct,
         SUM(away_match) AS Away_matches,
         SUM(away_win) AS Away_wins,
         ROUND(100.0 *SUM(away_win) / NULLIF(SUM(away_match), 0), 2) AS away_win_pct
     FROM (
         SELECT team1 AS team,
                1 AS Home_match,
                CASE when winner = team1 then 1 else 0 end AS home_win,
                0 AS away_match,
                0 AS away_win
        FROM matches where winner is NOT NULL
        UNION ALL
        SELECT team2 AS team,
                0, 0,
                1 AS away_match,
                CASE when winner = team2 then 1 else 0 end AS away_win
        FROM matches where winner is NOT NULL
     )
GROUP BY team
ORDER BY home_win_pct DESC
""")

,team,Home_matches,Home_wins,home_win_pct,Away_matches,Away_wins,away_win_pct
0,Lucknow Super Giants,22,16,72.73,21,8,38.10
1,Rising Pune Supergiant,7,5,71.43,9,5,55.56
2,Chennai Super Kings,128,75,58.59,109,63,57.80
3,Mumbai Indians,123,70,56.91,138,74,53.62
4,Delhi Capitals,41,23,56.10,50,25,50.00
5,Rajasthan Royals,101,55,54.46,118,57,48.31
6,Kolkata Knight Riders,121,65,53.72,130,66,50.77
7,Gujarat Titans,21,11,52.38,24,17,70.83
8,Sunrisers Hyderabad,86,44,51.16,96,44,45.83
9,Royal Challengers Bangalore,132,66,50.00,105,50,47.62


**Key Insights**

Lucknow Super Giants have the highest home win rate (72.73%) but drop drastically
to 38.10% away — making them the most home-dependent team in the IPL, suggesting
their performance is heavily influenced by home crowd and familiar conditions.
CSK and Mumbai Indians are the most consistent teams overall — both maintaining
50%+ win rates at home and away across 100+ matches, proving they are true
powerhouse franchises regardless of venue.
Gujarat Titans are a rare exception — they actually perform better away (70.83%)
than at home (52.38%), making them the only team in the IPL that thrives more
as a visiting side.